In [ ]:
import time
import numpy as np
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from executor.pauli_propagation import PauliPropagationExecutor
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# ===== Data Loading =====

def load_data(n_qubits, n_samples=None, test_size=0.2, random_state=42):
    digits = load_digits()
    X, y = digits.data, digits.target
    
    mask = (y == 0) | (y == 1)
    X, y = X[mask], y[mask]
    
    if n_samples is not None:
        idx = np.random.default_rng(random_state).choice(len(X), min(n_samples, len(X)), replace=False)
        X, y = X[idx], y[idx]
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    
    if n_qubits < 64:
        pca = PCA(n_components=n_qubits, random_state=random_state)
        X_train = pca.fit_transform(X_train)
        X_test = pca.transform(X_test)
    
    X_min, X_max = X_train.min(), X_train.max()
    X_train = (X_train - X_min) / (X_max - X_min) * np.pi
    X_test = (X_test - X_min) / (X_max - X_min) * np.pi
    
    return X_train, X_test, y_train, y_test

# ===== Config =====

n_qubits = 64
n_layers = 2
n_samples = 100
max_weight = 3

executor = PauliPropagationExecutor(max_weight=max_weight)


X_train, X_test, y_train, y_test = load_data(n_qubits, n_samples=n_samples)

data_params = ParameterVector('x', n_qubits)

# Fixed random parameters (no trainable params)

rng = np.random.default_rng(42)
fixed_angles = rng.uniform(0, 2 * np.pi, size=n_layers * n_qubits)

def build_circuit():
    """Circuit for fast Pauli propagation: RZ data encoding + fixed RY + CZ entanglement."""
    circuit = QuantumCircuit(n_qubits)
    
    for layer in range(n_layers):
        # Data encoding via RZ
        for q in range(n_qubits):
            circuit.rz(data_params[q], q)
        
        # Fixed random RY rotations
        for q in range(n_qubits):
            circuit.ry(fixed_angles[layer * n_qubits + q], q)
        
        # CZ entanglement (Clifford - doesn't expand Pauli weight)
        for q in range(n_qubits):
            circuit.cz(q, (q + 1) % n_qubits)
    
    return circuit

circuit = build_circuit()

# ===== Observables: single-qubit X, Y, Z on all qubits =====

def build_observables():
    observables = []
    for pauli in ['X', 'Y', 'Z']:
        for i in range(n_qubits):
            pauli_str = 'I' * i + pauli + 'I' * (n_qubits - i - 1)
            observables.append(SparsePauliOp([pauli_str], coeffs=[1.0]))
    return observables

observables = build_observables()
n_features = len(observables)  # 3 * n_qubits

# ===== Feature map via Pauli Propagation =====

def make_params(x):
    return {f'x[{i}]': x[i] for i in range(n_qubits)}

def compute_features(X_data):
    """Compute projected quantum kernel features: φ(x) = [⟨O_1⟩_x, ..., ⟨O_k⟩_x]"""
    features = np.zeros((len(X_data), n_features))
    for i, x in enumerate(X_data):
        start = time.perf_counter()
        params = make_params(x)
        for j, obs in enumerate(observables):
            features[i, j] = executor.expectation_value(circuit, obs, **params)
        print(f"Computed features for sample {i+1}/{len(X_data)} in {time.perf_counter() - start:.2f}s")
    return features

print("Computing projected quantum kernel features...")
t0 = time.perf_counter()
X_train_features = compute_features(X_train)
X_test_features = compute_features(X_test)
feature_time = time.perf_counter() - t0
print(f"Feature computation: {feature_time:.2f}s")

# ===== SVM with projected quantum kernel =====

# PQK: K(x, x') = φ(x)·φ(x') = Σ_i ⟨O_i⟩_x ⟨O_i⟩_x'

print("Training SVM with projected quantum kernel...")
t0 = time.perf_counter()
svm = SVC(kernel='linear')  # linear kernel on quantum features = PQK
svm.fit(X_train_features, y_train)
train_time = time.perf_counter() - t0

y_pred_train = svm.predict(X_train_features)
y_pred_test = svm.predict(X_test_features)

acc_train = accuracy_score(y_train, y_pred_train)
acc_test = accuracy_score(y_test, y_pred_test)

# ===== Results =====

print("\n" + "="*70)
print("PROJECTED QUANTUM KERNEL SVM (Pauli Propagation)")
print("="*70)
print(f"Circuit: {n_qubits} qubits, {n_layers} layers, {len(circuit.data)} gates")
print(f"Observables: {n_features} (X, Y, Z on each qubit)")
print(f"Dataset: {len(X_train)} train, {len(X_test)} test samples")
print("-"*70)
print(f"Feature computation time: {feature_time:.2f}s")
print(f"SVM training time:        {train_time:.4f}s")
print(f"Train accuracy:           {acc_train*100:.1f}%")
print(f"Test accuracy:            {acc_test*100:.1f}%")
print("="*70)

Computing projected quantum kernel features...
Computed features for sample 1/80 in 3.35s
Computed features for sample 2/80 in 3.31s
Computed features for sample 3/80 in 3.30s
Computed features for sample 4/80 in 3.58s


KeyboardInterrupt: 